# US 5.1 -- Uncertainty Quantification (MC Dropout)

The first step in active domain adaptation is figuring out **which** target-domain
samples the model is most uncertain about.  If the model already predicts an image
confidently (all pixels near 0 or 1), labeling that image will not teach it much.
We want the images where the model is confused.

We use **MC Dropout** (Monte Carlo Dropout) to estimate uncertainty.  The idea:

1. Keep dropout **enabled** at inference time (normally it is turned off).
2. Run T forward passes on the same image.  Each pass randomly drops different
   neurons, so we get T slightly different predictions.
3. Compute the **entropy** of the averaged predictions.  High entropy = high
   uncertainty.

## What you will learn

1. How MC Dropout works (intuition and math)
2. How to enable MC Dropout on a DANN model
3. How to compute the probability stack and entropy map
4. How to visualise uncertainty maps
5. How to use the CLI for batch uncertainty computation

## CLI equivalent

```bash
udm-epic5 uncertainty --config configs/epic5_active.yaml --T 20
```

## Prerequisites

- Python 3.12, PyTorch
- Install: `uv pip install -e ".[epic5]"`
- A trained DANN checkpoint (from Epic 4) or use `pretrained=False` for demos

---
## 1. How MC Dropout Works

### Standard inference vs MC Dropout

| Step | Standard | MC Dropout |
|------|----------|------------|
| Dropout at test time | Off (`model.eval()`) | **On** for dropout layers only |
| Number of forward passes | 1 | T (e.g., 20) |
| Output | Single prediction | T predictions |
| Uncertainty | Not available | Computed from variation across T passes |

### The math

Given T forward passes, we get T probability maps $\hat{p}_1, \hat{p}_2, \dots, \hat{p}_T$
for each pixel.  The mean prediction is:

$$\bar{p} = \frac{1}{T} \sum_{t=1}^{T} \hat{p}_t$$

The **predictive entropy** measures how uncertain the averaged prediction is:

$$H(\bar{p}) = -\bar{p} \log_2(\bar{p}) - (1 - \bar{p}) \log_2(1 - \bar{p})$$

- $H = 0$ when $\bar{p}$ is near 0 or 1 (confident prediction)
- $H = 1$ when $\bar{p} = 0.5$ (maximum uncertainty for binary classification)

For an **image-level** uncertainty score, we take the mean entropy across all pixels:

$$U(x) = \frac{1}{HW} \sum_{h,w} H(\bar{p}_{h,w})$$

### Why it works

Gal & Ghahramani (2016) showed that MC Dropout is equivalent to approximate
Bayesian inference.  Each dropout mask samples a different sub-network from
the posterior distribution over weights.  The disagreement between sub-networks
is a principled estimate of **epistemic uncertainty** (uncertainty due to
limited training data).

---
## 2. Setup: Enable MC Dropout on a DANN Model

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from udm_epic4.models.dann import DANNModel
from udm_epic5.uncertainty.mc_dropout import (
    enable_mc_dropout,
    mc_dropout_uncertainty,
    compute_entropy,
)

In [ ]:
# Create and prepare the model
model = DANNModel(
    backbone="convnext_tiny",
    pretrained=False,
    in_chans=3,
    decoder_channels=[256, 128, 64, 32],
    domain_head_hidden=256,
)

# In a real workflow you would load a trained checkpoint:
# model.load_state_dict(torch.load("checkpoints/dann_best.pt"))

# Enable MC Dropout: sets model to eval mode, then switches dropout layers
# back to train mode so they remain active during inference.
enable_mc_dropout(model)
print("MC Dropout enabled.")
print(f"Model training mode: {model.training}")  # False (eval)

# Verify dropout layers are active
dropout_layers = [(n, m) for n, m in model.named_modules() if 'drop' in n.lower()]
for name, mod in dropout_layers:
    print(f"  {name}: training={mod.training}")  # True (active)

---
## 3. Computing the Probability Stack

We run T forward passes on the same input.  Each pass produces a different
probability map because different dropout masks are sampled.

The `mc_dropout_uncertainty` function handles this loop and returns a tensor
of shape `[T, B, 1, H, W]`.

In [ ]:
# Create a dummy batch of 2 images
dummy_batch = torch.randn(2, 3, 512, 512)
T = 20  # number of MC samples

with torch.no_grad():
    prob_stack = mc_dropout_uncertainty(model, dummy_batch, T=T)

print(f"Probability stack shape: {prob_stack.shape}")  # [20, 2, 1, 512, 512]
print(f"Value range: [{prob_stack.min():.3f}, {prob_stack.max():.3f}]")
print()

# Each of the T=20 passes gives a slightly different prediction.
# Let's look at the variation for the first image, center pixel:
center_h, center_w = 256, 256
pixel_probs = prob_stack[:, 0, 0, center_h, center_w].numpy()
print(f"Center pixel probabilities across {T} passes:")
print(f"  Mean: {pixel_probs.mean():.4f}")
print(f"  Std:  {pixel_probs.std():.4f}")
print(f"  Min:  {pixel_probs.min():.4f}")
print(f"  Max:  {pixel_probs.max():.4f}")

---
## 4. Computing Entropy from the Probability Stack

The `compute_entropy` function:
1. Averages the T probability maps to get $\bar{p}$.
2. Computes binary entropy $H(\bar{p})$ at each pixel.
3. Returns an entropy map of shape `[B, H, W]`.

In [ ]:
# Compute per-pixel entropy
entropy_map = compute_entropy(prob_stack)  # [B, H, W]

print(f"Entropy map shape: {entropy_map.shape}")  # [2, 512, 512]
print(f"Entropy range: [{entropy_map.min():.4f}, {entropy_map.max():.4f}]")
print()

# Image-level uncertainty score: mean entropy across all pixels
for i in range(entropy_map.shape[0]):
    img_uncertainty = entropy_map[i].mean().item()
    print(f"  Image {i}: mean entropy = {img_uncertainty:.4f}")

print()
print("Higher mean entropy => more uncertain => better candidate for labeling.")

---
## 5. Visualising Uncertainty Maps

Uncertainty maps show *where* in the image the model is confused.  Bright
regions indicate high entropy (the model gives different predictions across
MC passes).  This is useful for understanding **what** the model finds hard.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for row in range(2):
    # Original image (first channel, denormalised for display)
    img = dummy_batch[row, 0].numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'Image {row}', fontsize=11)
    axes[row, 0].axis('off')

    # Mean prediction
    mean_pred = prob_stack[:, row, 0].mean(dim=0).numpy()
    axes[row, 1].imshow(mean_pred, cmap='RdYlBu_r', vmin=0, vmax=1)
    axes[row, 1].set_title(f'Mean prediction', fontsize=11)
    axes[row, 1].axis('off')

    # Entropy map
    ent = entropy_map[row].numpy()
    im = axes[row, 2].imshow(ent, cmap='hot', vmin=0, vmax=1)
    axes[row, 2].set_title(f'Entropy (uncertainty)', fontsize=11)
    axes[row, 2].axis('off')

fig.colorbar(im, ax=axes[:, 2], label='Entropy', shrink=0.8)
fig.suptitle('MC Dropout Uncertainty Maps', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

print("Interpretation:")
print("  - Dark regions in the entropy map: model is confident (low uncertainty)")
print("  - Bright regions: model disagrees across MC passes (high uncertainty)")
print("  - Boundary regions often show higher uncertainty than interiors.")

In [ ]:
# Distribution of per-pixel entropy values
fig, ax = plt.subplots(figsize=(8, 4))

for i in range(entropy_map.shape[0]):
    ent_flat = entropy_map[i].numpy().flatten()
    ax.hist(ent_flat, bins=50, alpha=0.6, label=f'Image {i}', density=True)

ax.set_xlabel('Per-pixel entropy', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Distribution of Pixel-Level Uncertainty', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print("Most pixels should have low entropy (confident predictions).")
print("The tail of high-entropy pixels reveals the uncertain regions.")

---
## 6. Choosing T (Number of MC Passes)

More passes give more stable uncertainty estimates, but cost more compute.
Let's see how the image-level uncertainty score stabilises as T increases.

In [ ]:
# Sweep T from 2 to 50
single_image = dummy_batch[:1]  # [1, 3, 512, 512]
T_values = [2, 5, 10, 15, 20, 30, 40, 50]
mean_entropies = []

for t in T_values:
    with torch.no_grad():
        ps = mc_dropout_uncertainty(model, single_image, T=t)
        ent = compute_entropy(ps)
        mean_entropies.append(ent.mean().item())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(T_values, mean_entropies, 'o-', linewidth=2, color='#2196F3')
ax.set_xlabel('T (number of MC passes)', fontsize=12)
ax.set_ylabel('Mean entropy', fontsize=12)
ax.set_title('Uncertainty Estimate vs Number of MC Passes', fontsize=14)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print("The estimate stabilises around T=20.")
print("We use T=20 as the default in the CLI and configs.")

---
## 7. CLI: Batch Uncertainty Computation

For a full dataset, use the CLI:

```bash
udm-epic5 uncertainty \
    --config configs/epic5_active.yaml \
    --checkpoint checkpoints/dann_best.pt \
    --T 20 \
    --output results/uncertainty_scores.csv
```

This will:
1. Load the DANN model from the checkpoint.
2. Enable MC Dropout.
3. Run T=20 passes on every target-domain image.
4. Save a CSV with columns: `image_path`, `mean_entropy`, `max_entropy`.

The CSV is consumed by the selection step (`udm-epic5 select`).

---
## Summary

- MC Dropout keeps dropout active during inference to estimate uncertainty.
- T forward passes produce T probability maps; their entropy measures uncertainty.
- Image-level score = mean per-pixel entropy.
- T=20 gives stable estimates; higher T is diminishing returns.
- Use `udm-epic5 uncertainty` for batch processing.

**Next:** `epic5_02_diversity.ipynb` -- diversity-based sample selection to
complement uncertainty.